# Domain 4: Prompt Engineering & Structured Output
## Task Statement 4.6: Multi-Instance & Multi-Pass Review Architectures
**Certification Weight**: 18% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand how to eliminate author bias through independent Claude review instances and implement a two-pass review architecture (per-file local pass + cross-file integration pass).

---

## 🏗️ Architecture: Multi-Instance Two-Pass Review Pipeline

```mermaid
flowchart TD
    Generator["Agent 1:<br>Code Generator Instance"] --> Codebase["Generated Code Artifacts<br>(auth.py, client.py)"]
    
    subgraph Pass1["Pass 1: File-Level Reviewers (Isolated)"]
        ReviewerA["Claude Instance A<br>(Reviews auth.py)"]
        ReviewerB["Claude Instance B<br>(Reviews client.py)"]
    end
    
    Codebase --> ReviewerA
    Codebase --> ReviewerB
    
    ReviewerA --> FileFindingsA["Local Issues:<br>Off-by-one, typing, edge cases"]
    ReviewerB --> FileFindingsB["Local Issues:<br>Error handling, logging"]
    
    subgraph Pass2["Pass 2: System Architecture Reviewer"]
        CrossReviewer["Claude Instance C<br>(Cross-File Integration)<br>Reviews interactions & API contracts"]
    end
    
    Codebase --> CrossReviewer
    CrossReviewer --> ContractFindings["Contract Mismatches:<br>Token expiration mismatch,<br>parameter name discrepancies"]
    
    FileFindingsA --> Aggregator["Synthesis Aggregator"]
    FileFindingsB --> Aggregator
    ContractFindings --> Aggregator
    
    Aggregator --> FinalAudit["Consolidated, Deduplicated<br>Audit Report"]
```

---

## ⚠️ Anti-Patterns vs Best Practices

| Anti-Pattern (Fails on Exam) | Recommended Architectural Practice |
|---|---|
| **Author Self-Review**: The same agent generates code and then in the same prompt is asked "Now check if there are bugs in your code". (Symptom: confirmation bias). | **Independent Fresh Instances**: Spawn fresh review instances with zero generation memory. |
| **Monolithic Single Pass**: Reviewing a 20-file PR in a single prompt. (Symptom: misses both local syntax bugs and cross-module interface drifts). | **Two-Pass Architecture**: Pass 1 reviews individual files for micro-correctness; Pass 2 reviews cross-file contracts. |

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Setup
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 4.6 Multi-Pass Review Architectures.")

### 🔍 Step 2: Multi-Pass Pipeline Implementation
We implement Pass 1 (local inspection) and Pass 2 (cross-file interface consistency).

In [ ]:
# Step 2: Multi-Pass Pipeline
@dataclass
class ReviewFinding:
    pass_number: int
    rule: str
    target: str
    message: str


class IndependentReviewerInstance:
    @staticmethod
    def review_file(filename: str, content: str) -> List[ReviewFinding]:
        """Pass 1: Local isolated checks."""
        findings = []
        if "except:" in content or "except Exception: pass" in content:
            findings.append(ReviewFinding(
                pass_number=1,
                rule="NO_BARE_EXCEPT",
                target=filename,
                message="Bare except block detected. Specific exceptions must be handled.",
            ))
        return findings


class MultiPassReviewPipeline:
    def __init__(self) -> None:
        self.reviewer = IndependentReviewerInstance()

    def run_review(self, codebase: Dict[str, str]) -> Dict[str, Any]:
        all_findings = []

        # Pass 1: Local File Reviews
        for fname, content in codebase.items():
            all_findings.extend(self.reviewer.review_file(fname, content))

        # Pass 2: Integration Signature Cross-Check
        # Check if caller uses argument name that matches callee definition
        if "auth_api.py" in codebase and "auth_client.py" in codebase:
            api_code = codebase["auth_api.py"]
            client_code = codebase["auth_client.py"]

            # If api defines 'bearer_token' but client passes 'auth_token='
            if "bearer_token:" in api_code and "auth_token=" in client_code:
                all_findings.append(ReviewFinding(
                    pass_number=2,
                    rule="INTERFACE_PARAMETER_MISMATCH",
                    target="auth_client.py -> auth_api.py",
                    message="Call site passes 'auth_token' but target function defines 'bearer_token'.",
                ))

        pass1_count = sum(1 for f in all_findings if f.pass_number == 1)
        pass2_count = sum(1 for f in all_findings if f.pass_number == 2)
        return {
            "approved": len(all_findings) == 0,
            "pass_1_local_count": pass1_count,
            "pass_2_integration_count": pass2_count,
            "findings": all_findings,
        }

print(" MultiPassReviewPipeline initialized.")

### 🧪 Step 3: Interactive Simulation
We test a codebase containing:
- `auth_api.py`: Has bare except (caught in Pass 1).
- `auth_client.py`: Calls `verify_session(auth_token=...)` when API expects `bearer_token` (caught in Pass 2).

In [ ]:
# Step 3: Simulation
codebase_sample = {
    "auth_api.py": "def verify_session(bearer_token: str):\n    try:\n        pass\n    except:\n        return False",
    "auth_client.py": "def login():\n    return verify_session(auth_token='jwt_xyz')",
}

pipeline = MultiPassReviewPipeline()
report = pipeline.run_review(codebase_sample)

print("Review Approved:", report["approved"])
print(f"Pass 1 Local Findings: {report['pass_1_local_count']}")
print(f"Pass 2 Integration Findings: {report['pass_2_integration_count']}")
for f in report["findings"]:
    print(f"  [Pass {f.pass_number}] {f.rule} in {f.target}: {f.message}")

### 🔬 Step 4: Verification Assertions
We assert that:
1. Pass 1 flags local bare except.
2. Pass 2 flags cross-module keyword argument mismatch.
3. Review fails approval.

In [ ]:
# Step 4: Verification Checks
assert report["approved"] is False
assert report["pass_1_local_count"] >= 1
assert report["pass_2_integration_count"] >= 1

rules = [f.rule for f in report["findings"]]
assert "NO_BARE_EXCEPT" in rules
assert "INTERFACE_PARAMETER_MISMATCH" in rules

print(" All verification assertions passed! Multi-pass review verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Local per-file reviews (Pass 1) **cannot** detect interface mismatches between callers and callees.
- **Rule 2**: Cross-file integration reviews (Pass 2) are required to verify contract signatures and shared state.
- **Rule 3**: Combine isolated passes to prevent cognitive overload while catching multi-module regressions.